In [1]:
import skimage as sk
import os
import numpy as np
import pandas as pd
import re
from cellpose import models, io
from random import randint
from glob import glob
import napari

In [ ]:
def get_center_project(file):
    img = sk.io.imread(file)
    center_z = img.shape[1]//2
    mid_slices = img[:,center_z-1:center_z+2,...]
    median_proj = np.median(mid_slices,axis=1)
    if img.shape[-1] == 1024:
        md_gauss = sk.filters.gaussian(median_proj,2)
    else:
        md_gauss = sk.filters.gaussian(median_proj,1)
    return median_proj, md_gauss, median_proj[0,...]

def process_nuc_channel(nuc_channel):
    sigma = nuc_channel.shape[-1]//3
    gauss = sk.filters.gaussian(nuc_channel,sigma)
    norm = sk.exposure.rescale_intensity(nuc_channel / gauss,out_range=(0,256))
    if nuc_channel.shape[-1] == 1024:
        sm_gauss = sk.filters.gaussian(norm,sigma=1)
        footprint = sk.morphology.disk(14)
        top_hat = sk.morphology.white_tophat(sm_gauss,footprint)
        md_gauss = sk.filters.gaussian(top_hat,4)
    else:
        sm_gauss = sk.filters.gaussian(norm,sigma=0.5)
        footprint = sk.morphology.disk(7)
        top_hat = sk.morphology.white_tophat(sm_gauss,footprint)
        md_gauss = sk.filters.gaussian(top_hat,2)        
    return md_gauss

def run_cp(img,cell_prob=0.0,flow=0.4,size=15):
    if img.shape[-1] == 1024:
        masks, _, _ = model.eval(img, rescale=0.5, resample=True, normalize=True, flow_threshold=flow, cellprob_threshold=cell_prob,min_size=size)
    else:
        masks, _, _ = model.eval(img, normalize=True, flow_threshold=flow, cellprob_threshold=cell_prob,min_size=size)
    return masks

def _filter_cells(cleared_cells,relabled_nuc):
    cell_props = sk.measure.regionprops(cleared_cells,relabled_nuc)
    cells_no_nuc = []
    cell_indicies = np.arange(np.max(cleared_cells)+1)
    for c in cell_props:
        label = c.label
        if c.intensity_max == 0:
            cells_no_nuc.append(label)
    cells_to_mask = np.isin(cell_indicies, cells_no_nuc)
    cells_mask = cells_to_mask[cleared_cells.reshape(-1)].reshape(cleared_cells.shape)
    cleared_cells[cells_mask] = 0
    relabled_cells, _, _ = sk.segmentation.relabel_sequential(cleared_cells)
    return relabled_cells

def clear_object_boarders(cell_masks,nuc_masks,name,path):
    cleared_cells, _, _ = sk.segmentation.relabel_sequential(sk.segmentation.clear_border(cell_masks))
    binary_cells = np.array(cleared_cells > 0,dtype="bool")
    masked_nuc = nuc_masks * binary_cells
    masked_nuc[masked_nuc>0] = 1
    matched_cells_to_nuc = _filter_cells(cleared_cells,masked_nuc)
    cell_bounaries = sk.segmentation.find_boundaries(matched_cells_to_nuc)
    masked_boundaries = np.array(matched_cells_to_nuc > 0,dtype='bool')
    masked_boundaries[cell_bounaries] = False
    if cell_masks.shape[-1] == 1024:
        footprint = sk.morphology.disk(2)
    else:
        footprint = sk.morphology.disk(1)
    eroded_masked_boundaries = sk.morphology.erosion(masked_boundaries,footprint)
    relabeled_nuc = (masked_nuc * eroded_masked_boundaries)*matched_cells_to_nuc # using eroded cells to crop nuclei within the cell boundary and minimize/remove overlaping nuclei labels within neighboring cells
    sk.io.imsave(os.path.join(path,f"{name}_cell_masks.tif"),matched_cells_to_nuc,check_contrast=False)
    sk.io.imsave(os.path.join(path,f"{name}_nuclei_masks.tif"),relabeled_nuc,check_contrast=False)
    return matched_cells_to_nuc, relabeled_nuc


def cyto_objects(cells, nuc, name,path):
    label_cyto = cells - nuc
    sk.io.imsave(os.path.join(path,f"{name}_cyto_masks.tif"),label_cyto,check_contrast=False)
    return label_cyto

def rename(x):
    pattern_1 = re.compile(r"^(?P<cell>....)_(?P<condition>DMSO|drug)_(?P<time_point>\d+)_")
    pattern_2 = re.compile(r"^(?P<cell>....)_(?P<time_point>\d+)\w+_(?P<condition>DMSO|drug)_")
    if re.match(pattern_1,x):
        return f"{re.search(pattern_1,x).group('cell').lower()}_{re.search(pattern_1,x).group('condition').lower()}_{re.search(pattern_1,x).group('time_point')}min"
    elif re.match(pattern_2,x):
        return f"{re.search(pattern_2,x).group('cell').lower()}_{re.search(pattern_2,x).group('condition').lower()}_{re.search(pattern_2,x).group('time_point')}min"
    else:
        return "No_Match"

def cell_type(x):
    if 'hela' in x:
        return 'hela'
    if 'pc12' in x:
        return 'pc12'

def condition(x):
    if 'drug' in x:
        return 'drug'
    if 'dmso' in x:
        return 'dmso'

def get_measurements(label_cyto,label_cells,label_nuc,img,name):
    props = ["label","intensity_median"]
    reshape_img = np.moveaxis(img,0,-1)
    cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_img,properties=props))
    nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_nuc,reshape_img,properties=props))
    cell_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cells,reshape_img,properties=props))
    cyto_props = cyto_props.add_prefix("Cyto_")
    nuc_props = nuc_props.add_prefix("Nuc_")
    cell_props = cell_props.add_prefix("Cell_")
    cyto_props["Nur77_Nuc_Cyto_Ratio"] = nuc_props["Nuc_intensity_median-1"]/cyto_props["Cyto_intensity_median-1"]
    merged = pd.concat([cell_props,nuc_props,cyto_props],axis=1)
    merged['Image'] = name
    merged['Clean_Name'] = merged['Image'].apply(rename)
    merged['Cell_Type'] = merged['Clean_Name'].apply(cell_type)
    merged['Treatment'] = merged['Clean_Name'].apply(condition)
    return merged


In [ ]:
files = sorted(glob("/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Exported_Tiffs/*.tif"))
names = [os.path.basename(f)[:-4] for f in files]
model = models.CellposeModel(gpu=True)

In [ ]:
masks_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Output_Masks/"
measurements_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/"
dfs = []
for f, n in zip(files, names):
    print(f"processing {n}")
    img = sk.io.imread(f)
    if len(img.shape) == 4:
        median_proj, median_filter, nuc_channel = get_center_project(f)
        nuc_norm = process_nuc_channel(nuc_channel)
        cell_masks = run_cp(median_filter,flow=0.6,size=100)
        nuc_masks = run_cp(nuc_norm,cell_prob=-1.0,flow=0.6)
        cleared_cells, cleared_nuc = clear_object_boarders(cell_masks,nuc_masks,n,masks_save_path)
        cyto_masks = cyto_objects(cleared_cells, cleared_nuc,n,masks_save_path)
        df = get_measurements(cyto_masks,cleared_cells,cleared_nuc,median_proj,n)
        dfs.append(df)
    else:
        print(f"Skipping {n}, has shape {img.shape}")        
all_data = pd.concat(dfs,ignore_index=True)
all_data.to_csv(os.path.join(measurements_path,'Nur77_NR4A1_Translocation_Ratio_Measurements.csv'),index=False)

In [ ]:
all_data['Group'] = all_data['Image'].apply(rename)

In [ ]:
all_data['Cell_Type'] = all_data['Group'].apply(cell_type)
all_data['Treatment'] = all_data['Group'].apply(condition)

In [ ]:
all_data.to_csv(os.path.join(measurements_path,'Nur77_NR4A1_Translocation_Ratio_Measurements.csv'),index=False)

In [ ]:
all_data

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
all_data_filtered = all_data.query('Nur77_Nuc_Cyto_Ratio != inf')

In [ ]:
all_data_filtered.sort_values("Cell_Type")

In [ ]:
sns.swarmplot(all_data_filtered,x='Group', y='Nur77_Nuc_Cyto_Ratio', hue='Cell_Type')

In [ ]:
img_medians = all_data_filtered.groupby(['Image','Group','Cell_Type','Treatment'],as_index=False).median(numeric_only=True)

In [ ]:
medians = all_data_filtered.groupby(['Group','Cell_Type','Treatment'],as_index=False).median(numeric_only=True)

In [ ]:
sns.barplot(medians,x='Group', y='Nur77_Nuc_Cyto_Ratio', hue='Cell_Type',hue_order=medians['Cell_Type'])

In [ ]:
hela_groups = medians.query('Cell_Type == "hela"')
pc12_groups = medians.query('Cell_Type == "pc12"')

In [ ]:
hela_groups

In [ ]:
fig, axs = plt.subplots(2,1)
sns.barplot(hela_groups,x='Group', y='Nur77_Nuc_Cyto_Ratio', hue='Treatment',hue_order=medians['Treatment'], ax=axs[0])
sns.barplot(pc12_groups,x='Group', y='Nur77_Nuc_Cyto_Ratio', hue='Treatment',hue_order=medians['Treatment'], ax=axs[1])


In [ ]:
i = randint(0,len(files))
print(i)
test_img = sk.io.imread(files[i])
test_img.shape

In [ ]:
center_z = test_img.shape[1]//2

In [ ]:
mid_slices = test_img[:,center_z-1:center_z+2,...]

In [ ]:
mid_slices.shape

In [ ]:
#i = randint(0,len(files))

n = 'test_center_proj_and_norm'
test_center_proj, median_cent, nuc_channel = get_center_project(files[i])
test_nuc_norm = process_nuc_channel(nuc_channel)
test_cell_masks = run_cp(median_cent,flow=0.6,size=100)
test_nuc_masks = run_cp(test_nuc_norm,cell_prob=-1.0,flow=0.6)


In [ ]:
#test_object_boarders = sk.segmentation.find_boundaries(test_cell_masks,mode="inner")
test_cleared_cells, test_cleared_nuc = clear_object_boarders(test_cell_masks,test_nuc_masks,n,masks_save_path)
test_label_cyto = cyto_objects(test_cleared_cells, test_cleared_nuc,n,masks_save_path)

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(test_center_proj,channel_axis=0)
v.add_image(test_nuc_norm)
v.add_labels(test_cleared_cells)
v.add_labels(test_cleared_nuc)
#v.add_labels(eroded_masked_boundaries)
v.add_labels(test_label_cyto)

In [ ]:
cyto_props, nuc_props, cell_props,merged = get_measurements(test_label_cyto,test_cleared_cells,test_cleared_nuc,median_cent,n)

In [ ]:
merged_test = pd.concat([cyto_props, nuc_props, cell_props],axis=1)

In [ ]:
merged_test

In [ ]:
v.add_image(test_img,channel_axis=0)

In [ ]:
join, map1, map2 = sk.segmentation.join_segmentations(test_cleared_cells,test_cleared_nuc,return_mapping=True)

In [ ]:
v.add_labels(join)

In [ ]:
#Need to add in preprocessing to remove background DAPI interferring with segmentation
# Testing adaptive hist
test_img = sk.io.imread(files[i])
#sigma = test_center_proj.shape[-1]//3
adapt_hist_proj = sk.exposure.equalize_adapthist(test_center_proj/255.0)

In [ ]:
# Testing background subtraction with rolling ball
background = sk.restoration.rolling_ball(test_center_proj[0],radius=5)
background_sub = test_center_proj[0] - background

In [ ]:
# Testing background norm with gaussian
sigma = test_center_proj.shape[-1]//3
gauss = sk.filters.gaussian(test_center_proj[0],sigma)
norm = sk.exposure.rescale_intensity(test_center_proj[0] / gauss,out_range=(0,256))

In [ ]:
sm_gauss = sk.filters.gaussian(norm,sigma=1)

In [ ]:
footprint = sk.morphology.disk(12)
top_hat = sk.morphology.white_tophat(sm_gauss,footprint)

In [ ]:
v.add_image(top_hat,name='tophat_12')

In [ ]:
mask, _, _ = model.eval(top_hat,normalize=True)

In [ ]:
v.add_labels(mask,name='cp_mask_tophat_12')

In [ ]:
#threshold and mask on norm
mask = sm_gauss > sk.filters.threshold_multiotsu(sm_gauss,classes=5)[-2]


In [ ]:
v.add_image(sm_gauss)
v.add_labels(mask)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
all_thresh = sk.filters.try_all_threshold(norm,figsize=(10,6))
plt.show()

In [ ]:
v.add_image(mask)

In [ ]:
model = models.CellposeModel(gpu=True)
masks_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/Output_Masks/"
cyto_dfs = []
nuc_dfs = []
cell_dfs = []
for f, n in zip(files,names):
    center_proj = get_center_project(f)
    cell_masks = run_cp(center_proj)
    nuc_masks = run_cp(center_proj[0,...])
    object_boarders = sk.segmentation.find_boundaries(cell_masks,mode="inner")
    cleared_cells, cleared_nuc = clear_object_boarders(cell_masks,nuc_masks,n,masks_save_path)
    label_cyto = cyto_objects(cleared_cells, cleared_nuc, object_boarders,n,masks_save_path)
    cyto_props, nuc_props, cell_props = get_measurements(label_cyto,cleared_cells,cleared_nuc,center_proj,n)
    cyto_dfs.append(cyto_props)
    nuc_dfs.append(nuc_props)
    cell_dfs.append(cell_props)

df_save_path = "/DISKS/2TB/gkristin/Nur77_NR4A1_Translocation/"
merged_cyto = concat_measurements(cyto_dfs)
merged_cyto.to_csv(os.path.join(df_save_path,"Cytoplasm_Measurements.csv"))
merged_nuc = concat_measurements(nuc_dfs)
merged_nuc.to_csv(os.path.join(df_save_path,"Nuclei_Measurements.csv"))
merged_cell = concat_measurements(cell_dfs)
merged_cell.to_csv(os.path.join(df_save_path,"Cell_Measurements.csv"))


In [ ]:
i = randint(0,len(files))
img = sk.io.imread(files[i])
img.shape

In [ ]:
avg_img = np.mean(img,axis=1)
avg_img.shape

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(avg_img,channel_axis=0)

In [ ]:
model = models.CellposeModel(gpu=True)

In [ ]:
cell_masks, _, _ = model.eval(avg_img, normalize=True)
nuc_masks, _, _ = model.eval(avg_img[0,...], normalize=True)

In [ ]:
cell_masks, _, _ = model.eval(avg_img, normalize=True, rescale=0.5, resample=True, min_size=550)


In [ ]:
nuc_masks, _, _ = model.eval(avg_img[0,...], normalize=True, rescale=0.5, resample=True, min_size=550, cellprob_threshold=-1.0)

In [ ]:
v.add_labels(cell_masks)
v.add_labels(nuc_masks)

In [ ]:
object_boarders = sk.segmentation.find_boundaries(cell_masks,mode="inner")

In [ ]:
def clear_object_boarders(cell_masks,nuc_masks):
    cleared_cells, _, _ = sk.segmentation.relabel_sequential(sk.segmentation.clear_border(cell_masks))
    binary_cells = np.array(cleared_cells > 0,dtype="uint8")
    cleared_nuc, _, _ = sk.segmentation.relabel_sequential((binary_cells*nuc_masks))
    return cleared_cells, cleared_nuc

In [ ]:
def cyto_objects(cells, nuc, boundaries):
    binary_cells = np.array(cells>0,dtype='int8')
    binary_nuc = np.array(nuc>0,dtype='int8')
    erode_nuc = sk.morphology.erosion(binary_nuc,sk.morphology.footprint_rectangle((3,3)))
    #inter = (binary_cells - erode_nuc)
    binary_cyto = sk.exposure.rescale_intensity(((binary_cells - erode_nuc) - boundaries),in_range='image',out_range=(0,1)).astype(np.uint8)
    label_cyto = binary_cyto*cells
    return label_cyto

In [ ]:
def get_measurements(label_cyto,label_cells,label_nuc,img,name):
    props = ["label","intensity_median"]
    reshape_img = np.moveaxis(img,0,-1)
    cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_img,properties=props))
    nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_nuc,reshape_img,properties=props))
    cell_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cells,reshape_img,properties=props))
    cyto_props = cyto_props.add_prefix("Cyto_")
    nuc_props = nuc_props.add_prefix("Nuc_")
    cell_props = cell_props.add_prefix("Cell_")
    cyto_props["Nur77_Nuc_Cyto_Ratio"] = nuc_props["Nuc_intensity_median-1"]/cyto_props["Cyto_intensity_median-1"]
    cyto_props["Image"] = name
    nuc_props["Image"] = name
    cell_props["Image"] = name
    return cyto_props, nuc_props, cell_props

In [ ]:
def concat_measurements(dfs):
    return pd.concat(dfs,ignore_index=True)

In [ ]:
cleared_cells, cleared_nuc = clear_object_boarders(cell_masks,nuc_masks)

In [ ]:
label_cyto = cyto_objects(cleared_cells, cleared_nuc, object_boarders)

In [ ]:
v.add_labels(cleared_cells)
v.add_labels(cleared_nuc)
v.add_labels(label_cyto)

In [ ]:
df = get_measurements(label_cyto,cleared_cells,cleared_nuc,avg_img,names[i])

In [ ]:
df

In [ ]:
#measurements
props = ["label","intensity_median"]
reshape_avg = np.moveaxis(avg_img,0,-1)
cyto_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(label_cyto,reshape_avg,properties=["label","intensity_median"]))
nuc_props = pd.DataFrame.from_dict(sk.measure.regionprops_table(nuc_masks,reshape_avg,properties=["label","intensity_median"]))


In [ ]:
cyto_props = cyto_props.add_prefix("Cyto_")
nuc_props = nuc_props.add_prefix("Nuc_")

In [ ]:
nuc_props.columns

In [ ]:
import seaborn as sns

In [ ]:
sns.swarmplot(nuc_props,x="Nuc_intensity_median-1",hue="Nuc_label")

In [ ]:
nuc_ints_map = sk.util.map_array(nuc_masks,nuc_props["label"],nuc_props["intensity_median-1"])

In [ ]:
v.add_image(nuc_ints_map)